In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math
import numpy as np
from collections import Counter
from tqdm import tqdm
import os

In [2]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
# Constants
d_model = 512
num_heads = 8
num_layers = 4
d_ff = 2048
max_seq_len = 128
dropout_rate = 0.1
batch_size = 64
num_epochs = 15
learning_rate = 1e-4

In [4]:
# Special tokens
PAD_IDX = 0
UNK_IDX = 1
SOS_IDX = 2
EOS_IDX = 3

In [5]:
class Vocab:
    def __init__(self, min_freq=3):
        self.token_to_idx = {'<pad>': PAD_IDX, '<unk>': UNK_IDX, '<sos>': SOS_IDX, '<eos>': EOS_IDX}
        self.idx_to_token = {PAD_IDX: '<pad>', UNK_IDX: '<unk>', SOS_IDX: '<sos>', EOS_IDX: '<eos>'}
        self.idx = 4
        self.min_freq = min_freq
    
    def build_vocab(self, sentences):
        word_counts = Counter()
        for sentence in sentences:
            word_counts.update(sentence.split())
        
        # Add words that appear at least min_freq times
        for word, count in word_counts.items():
            if count >= self.min_freq and word not in self.token_to_idx:
                self.token_to_idx[word] = self.idx
                self.idx_to_token[self.idx] = word
                self.idx += 1
    
    def encode(self, sentence):
        return [self.token_to_idx.get(token, UNK_IDX) for token in sentence.split()]
    
    def decode(self, indices):
        return ' '.join([self.idx_to_token.get(idx, '<unk>') for idx in indices])
    
    def __len__(self):
        return len(self.token_to_idx)

In [6]:
# Dataset class
class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=max_seq_len):
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len
    
    def __len__(self):
        return len(self.src_sentences)
    
    def __getitem__(self, idx):
        src_sentence = self.src_sentences[idx]
        tgt_sentence = self.tgt_sentences[idx]
        
        # Encode sentences
        src_encoded = self.src_vocab.encode(src_sentence)
        tgt_encoded = self.tgt_vocab.encode(tgt_sentence)
        
        # Add SOS and EOS tokens
        tgt_input = [SOS_IDX] + tgt_encoded
        tgt_output = tgt_encoded + [EOS_IDX]
        
        # Pad or truncate
        src_encoded = self.pad_or_truncate(src_encoded)
        tgt_input = self.pad_or_truncate(tgt_input)
        tgt_output = self.pad_or_truncate(tgt_output)
        
        return torch.LongTensor(src_encoded), torch.LongTensor(tgt_input), torch.LongTensor(tgt_output)
    
    def pad_or_truncate(self, sequence):
        if len(sequence) > self.max_len:
            return sequence[:self.max_len]
        else:
            return sequence + [PAD_IDX] * (self.max_len - len(sequence))

In [7]:
# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=max_seq_len):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:x.size(0), :]

In [8]:
# Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout_rate)
    
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, V)
        return output
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        
        Q = self.w_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        attention_output = self.scaled_dot_product_attention(Q, K, V, mask)
        
        attention_output = attention_output.transpose(1, 2).contiguous().view(
            batch_size, -1, self.d_model
        )
        
        output = self.w_o(attention_output)
        return output

In [9]:
# Feed Forward Network
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout_rate)
    
    def forward(self, x):
        return self.linear2(self.dropout(torch.relu(self.linear1(x))))

In [10]:
# Encoder Layer
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout_rate)
    
    def forward(self, x, mask):
        # Self-attention
        attn_output = self.self_attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed forward
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        return x

In [11]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.cross_attention = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout_rate)
    
    def forward(self, x, enc_output, src_mask, tgt_mask):
        # Self-attention
        attn_output = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Cross-attention
        attn_output = self.cross_attention(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))
        
        # Feed forward
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        
        return x

In [12]:
# Transformer Model
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_len):
        super().__init__()
        self.d_model = d_model
        
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len)
        
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout_rate)
    
    def generate_mask(self, src, tgt):
        src_mask = (src != PAD_IDX).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != PAD_IDX).unsqueeze(1).unsqueeze(3)
        
        seq_len = tgt.size(1)
        nopeak_mask = (1 - torch.triu(torch.ones(1, seq_len, seq_len), diagonal=1)).bool()
        nopeak_mask = nopeak_mask.to(device)
        tgt_mask = tgt_mask & nopeak_mask
        
        return src_mask, tgt_mask
    
    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        
        # Embeddings and positional encoding
        src_embedded = self.dropout(self.positional_encoding(self.src_embedding(src) * math.sqrt(self.d_model)))
        tgt_embedded = self.dropout(self.positional_encoding(self.tgt_embedding(tgt) * math.sqrt(self.d_model)))
        
        # Encoder
        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)
        
        # Decoder
        dec_output = tgt_embedded
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask)
        
        # Final linear layer
        output = self.fc_out(dec_output)
        return output

In [13]:
def load_data(MAX_SENTENCES = 150000):
    src_file = "/kaggle/input/english-to-gujarati-machine-translation-dataset/en-gu/train.en"
    tgt_file = "/kaggle/input/english-to-gujarati-machine-translation-dataset/en-gu/train.gu"
    
    with open(src_file, 'r', encoding='utf-8') as f:
        src_sentences = [line.strip() for line in f.readlines()[:MAX_SENTENCES]]
    
    with open(tgt_file, 'r', encoding='utf-8') as f:
        tgt_sentences = [line.strip() for line in f.readlines()[:MAX_SENTENCES]]
    
    return src_sentences, tgt_sentences

In [14]:
def translate_sentence(model, sentence, src_vocab, tgt_vocab, max_len=50):
    model.eval()
    
    # Encode input sentence
    tokens = src_vocab.encode(sentence)
    tokens = tokens[:max_len]
    src_tensor = torch.LongTensor([tokens + [PAD_IDX] * (max_len - len(tokens))]).to(device)

    # Prepare masks and encoder output
    src_mask = (src_tensor != PAD_IDX).unsqueeze(1).unsqueeze(2)
    src_emb = model.dropout(model.positional_encoding(model.src_embedding(src_tensor) * math.sqrt(model.d_model)))
    memory = src_emb
    for layer in model.encoder_layers:
        memory = layer(memory, src_mask)

    # Start with <sos>
    tgt_indices = [SOS_IDX]
    
    for _ in range(max_len):
        tgt_tensor = torch.LongTensor([tgt_indices]).to(device)
        tgt_mask = (tgt_tensor != PAD_IDX).unsqueeze(1).unsqueeze(3)
        seq_len = tgt_tensor.size(1)
        nopeak_mask = (1 - torch.triu(torch.ones(1, seq_len, seq_len), diagonal=1)).bool().to(device)
        tgt_mask = tgt_mask & nopeak_mask

        tgt_emb = model.dropout(model.positional_encoding(model.tgt_embedding(tgt_tensor) * math.sqrt(model.d_model)))
        output = tgt_emb
        for layer in model.decoder_layers:
            output = layer(output, memory, src_mask, tgt_mask)

        output = model.fc_out(output)
        next_token = output[0, -1].argmax().item()
        if next_token == EOS_IDX:
            break
        tgt_indices.append(next_token)
    
    return tgt_vocab.decode(tgt_indices[1:])

In [15]:
# Main execution flow
print("Loading data...")
src_sentences, tgt_sentences = load_data()

# Build vocabularies
print("Building vocabularies...")
src_vocab = Vocab()
tgt_vocab = Vocab()

src_vocab.build_vocab(src_sentences)
tgt_vocab.build_vocab(tgt_sentences)

print(f"Source vocabulary size: {len(src_vocab)}")
print(f"Target vocabulary size: {len(tgt_vocab)}")

Loading data...
Building vocabularies...
Source vocabulary size: 32767
Target vocabulary size: 36070


In [16]:
# Split data (80% train, 20% validation)
split_idx = int(0.8 * len(src_sentences))
train_src, val_src = src_sentences[:split_idx], src_sentences[split_idx:]
train_tgt, val_tgt = tgt_sentences[:split_idx], tgt_sentences[split_idx:]

# Create datasets
train_dataset = TranslationDataset(train_src, train_tgt, src_vocab, tgt_vocab)
val_dataset = TranslationDataset(val_src, val_tgt, src_vocab, tgt_vocab)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [17]:
# Initialize model
model = Transformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers,
    d_ff=d_ff,
    max_seq_len=max_seq_len
).to(device)

# Initialize optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

In [18]:
print("Starting training...")

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    for batch_idx, (src, tgt_input, tgt_output) in enumerate(tqdm(train_loader)):
        src, tgt_input, tgt_output = src.to(device), tgt_input.to(device), tgt_output.to(device)
        
        optimizer.zero_grad()
        
        output = model(src, tgt_input)
        loss = criterion(output.reshape(-1, len(tgt_vocab)), tgt_output.reshape(-1))
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}")
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for src, tgt_input, tgt_output in val_loader:
            src, tgt_input, tgt_output = src.to(device), tgt_input.to(device), tgt_output.to(device)
            output = model(src, tgt_input)
            loss = criterion(output.reshape(-1, len(tgt_vocab)), tgt_output.reshape(-1))
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader)
    print(f"Validation Loss: {avg_val_loss:.4f}")

Starting training...


100%|██████████| 1875/1875 [14:32<00:00,  2.15it/s]


Epoch 1/15, Average Loss: 6.4595
Validation Loss: 5.7812


100%|██████████| 1875/1875 [14:42<00:00,  2.13it/s]


Epoch 2/15, Average Loss: 5.3919
Validation Loss: 5.2360


100%|██████████| 1875/1875 [14:41<00:00,  2.13it/s]


... (21 outputs left out)

100%|██████████| 1875/1875 [14:44<00:00,  2.12it/s]


Epoch 14/15, Average Loss: 1.9366
Validation Loss: 4.8185


100%|██████████| 1875/1875 [14:43<00:00,  2.12it/s]


Epoch 15/15, Average Loss: 1.7990
Validation Loss: 4.9132


In [19]:
# Test translation
test_sentences = ["I'm feeling very hungry", "Can I get your contact number?"]
for sent in test_sentences:
    translation = translate_sentence(model, sent, src_vocab, tgt_vocab)
    print(f"Input: {sent}")
    print(f"Translation: {translation}")
    print()

Input: I'm feeling very hungry
Translation: હું ખૂબ જ આઘાત લાગ્યો છું.

Input: Can I get your contact number?
Translation: હું તમારી હિંમત કરી શકો છો?



In [20]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 3.9 MB/s eta 0:00:00


In [21]:
# Evaluation metrics for translation model
import sacrebleu

def evaluate_model(model, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=50):
    model.eval()
    predictions = []
    references = []

    for src, tgt in zip(src_sentences, tgt_sentences):
        pred = translate_sentence(model, src, src_vocab, tgt_vocab, max_len=max_len)
        predictions.append(pred)
        references.append([tgt])

    # BLEU Score
    bleu = sacrebleu.corpus_bleu(predictions, references)
    print(f"\nBLEU Score: {bleu.score:.2f}")

    # CHRF Score
    chrf = sacrebleu.corpus_chrf(predictions, references)
    print(f"CHRF Score: {chrf.score:.2f}")

    # TER Score
    ter = sacrebleu.corpus_ter(predictions, references)
    print(f"TER Score: {ter.score:.2f}")

# Evaluate using validation set
print("\nEvaluating on validation set...")
evaluate_model(model, val_src[:100], val_tgt[:100], src_vocab, tgt_vocab)  # limit to 100 samples for speed


Evaluating on validation set...

BLEU Score: 7.47
CHRF Score: 28.74
TER Score: 113.51


In [22]:
# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'src_vocab': src_vocab,
    'tgt_vocab': tgt_vocab
}, 'transformer_en_gu.pth')